# Cleaning and Transforming Raw data and save as delta table.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window as W

In [0]:
%sql
-- create schema if not exists onehouse_dev_dtu_silver.silver_cleaned_schema

In [0]:
bronze_orders = spark.table("onehouse_dev_dtu_bronze.source_raw_schema.bronze_orders")

# Standardize values and convert data types
orders_cleaned = (
    bronze_orders
    .withColumn("customer_id", F.trim("customer_id"))
    .withColumn("product_id", F.trim("product_id"))
    .withColumn("status", F.initcap(F.trim("status")))
    .withColumn(
        "order_date",
        F.to_date("order_date", "yyyy-MM-dd")
    )
    .withColumn("quantity", F.col("quantity").cast("int"))
)

# Apply data quality rules
orders_checked = orders_cleaned.withColumn(
    "dq_error",
    F.when(F.col("order_id").isNull(), "Missing order_id")
    .when(F.col("customer_id").isNull() |
          (F.col("customer_id") == ""), "Invalid customer_id")
    .when(F.col("product_id").isNull() |
          (F.col("product_id") == ""), "Invalid product_id")
    .when(F.col("order_date").isNull(), "Invalid order_date")
    .when(F.col("quantity").isNull() |
          (F.col("quantity") <= 0), "Invalid quantity")
    .when(
        ~F.col("status").isin(
            "Completed", "Pending", "Cancelled"
        ) | F.col("status").isNull(),
        "Invalid status"
    )
)

# Separate valid and rejected records
valid_orders = orders_checked.filter(
    F.col("dq_error").isNull()
).drop("dq_error")

rejected_orders = orders_checked.filter(
    F.col("dq_error").isNotNull()
)

# Remove duplicate order IDs
# For production, use a defined latest-record rule if duplicates differ.
silver_orders = valid_orders.dropDuplicates(["order_id"])

# Save Silver and quarantine tables
silver_orders.write.format("delta").mode("overwrite") \
    .saveAsTable("onehouse_dev_dtu_silver.silver_cleaned_schema.silver_orders")

rejected_orders.write.format("delta").mode("overwrite") \
    .saveAsTable("onehouse_dev_dtu_silver.silver_cleaned_schema.quarantine_orders")

print("Silver orders and rejected records saved")

In [0]:
display(orders_checked)

In [0]:
%sql
SHOW TABLES IN onehouse_dev_dtu_silver.silver_cleaned_schema;

In [0]:
%sql
select * from onehouse_dev_dtu_silver.silver_cleaned_schema.silver_orders;

In [0]:
# CUSTOMERS
customers = spark.table("onehouse_dev_dtu_bronze.source_raw_schema.bronze_customers")

silver_customers = (
    customers
    .select(
        F.trim("customer_id").alias("customer_id"),
        F.trim("customer_name").alias("customer_name"),
        F.initcap(F.trim("region")).alias("region")
    )
    .filter(
        F.col("customer_id").isNotNull()
        & (F.col("customer_id") != "")
        & F.col("customer_name").isNotNull()
        & (F.col("customer_name") != "")
        & F.col("region").isNotNull()
        & (F.col("region") != "")
    )
    .dropDuplicates(["customer_id"])
)

# PRODUCTS
products = spark.table("onehouse_dev_dtu_bronze.source_raw_schema.bronze_products")

silver_products = (
    products
    .select(
        F.trim("product_id").alias("product_id"),
        F.trim("product_name").alias("product_name"),
        F.trim("category").alias("category"),
        F.col("unit_price").cast("decimal(18,2)").alias("unit_price")
    )
    .filter(
        F.col("product_id").isNotNull()
        & (F.col("product_id") != "")
        & F.col("product_name").isNotNull()
        & (F.col("product_name") != "")
        & F.col("unit_price").isNotNull()
        & (F.col("unit_price") >= 0)
    )
    .dropDuplicates(["product_id"])
)

# REGIONS
regions = spark.table("onehouse_dev_dtu_bronze.source_raw_schema.bronze_regions")

silver_regions = (
    regions
    .select(
        F.initcap(F.trim("region")).alias("region"),
        F.trim("region_manager").alias("region_manager")
    )
    .filter(
        F.col("region").isNotNull()
        & (F.col("region") != "")
        & F.col("region_manager").isNotNull()
        & (F.col("region_manager") != "")
    )
    .dropDuplicates(["region"])
)

# Save cleaned tables
silver_customers.write.format("delta").mode("overwrite") \
    .saveAsTable("onehouse_dev_dtu_silver.silver_cleaned_schema.silver_customers")

silver_products.write.format("delta").mode("overwrite") \
    .saveAsTable("onehouse_dev_dtu_silver.silver_cleaned_schema.silver_products")

silver_regions.write.format("delta").mode("overwrite") \
    .saveAsTable("onehouse_dev_dtu_silver.silver_cleaned_schema.silver_regions")

In [0]:
%skip
#Testing
bronze_orders_bad = spark.table("onehouse_dev_dtu_bronze.source_raw_schema.bad_records_bronze_orders")

# Standardize values and convert data types
bad_orders_cleaned = (
    bronze_orders_bad
    .withColumn("customer_id", F.trim("customer_id"))
    .withColumn("product_id", F.trim("product_id"))
    .withColumn("status", F.initcap(F.trim("status")))
    .withColumn(
        "order_date",
        F.try_to_date("order_date", "yyyy-MM-dd")
    )
    .withColumn("quantity", F.col("quantity").cast("int"))
)

# Apply data quality rules
bad_orders_checked = bad_orders_cleaned.withColumn(
    "dq_error",
    F.when(F.col("order_id").isNull(), "Missing order_id")
    .when(F.col("customer_id").isNull() |
          (F.col("customer_id") == ""), "Invalid customer_id")
    .when(F.col("product_id").isNull() |
          (F.col("product_id") == ""), "Invalid product_id")
    .when(F.col("order_date").isNull(), "Invalid order_date")
    .when(F.col("quantity").isNull() |
          (F.col("quantity") <= 0), "Invalid quantity")
    .when(
        ~F.col("status").isin(
            "Completed", "Pending", "Cancelled"
        ) | F.col("status").isNull(),
        "Invalid status"
    )
)

# Separate valid and rejected records  -- this syntax filter only null values in dq_error column, and drop others
valid_orders = bad_orders_checked.filter(
    F.col("dq_error").isNull()
).drop("dq_error")

rejected_orders = bad_orders_checked.filter(
    F.col("dq_error").isNotNull()
)

# Remove duplicate order IDs
# For production, use a defined latest-record rule if duplicates differ.
bad_silver_orders = valid_orders.dropDuplicates(["order_id"])

# # Save Silver and quarantine tables
# silver_orders.write.format("delta").mode("overwrite") \
#     .saveAsTable("onehouse_dev_dtu_silver.silver_cleaned_schema.silver_orders")

# rejected_orders.write.format("delta").mode("overwrite") \
#     .saveAsTable("onehouse_dev_dtu_silver.silver_cleaned_schema.quarantine_orders")

# print("Silver orders and rejected records saved")

In [0]:
%skip
display(rejected_orders)

In [0]:
%skip
display(bad_orders_cleaned)

In [0]:
%skip
display(bad_orders_checked)